# 01 — Data Loading and Validation

This notebook is the first of three notebooks in the pairs trading research pipeline. Its scope is limited to data loading, data validation, diagnostics inspection, and basic visualization of the loaded price and volume matrices.

The notebook calls the project’s Python modules rather than duplicating logic inside notebook cells. Specifically, it uses the data loading and validation functions defined in `src/` to fetch historical price and volume data, validate the resulting price and volume matrices, inspect diagnostics, save processed data for downstream notebooks, and plot normalized prices.

This notebook loads adjusted close and adjusted volume data from Tiingo.
The adjusted close matrix is validated by the price validator, and the adjusted
volume matrix is validated by the volume validator. A dollar-volume matrix is
created as adjusted close multiplied by adjusted volume, which will be used for
liquidity-aware pair selection in Notebook 02.

The full project notebook flow is:

1. `01_data_loading_validation.ipynb` — load and validate market data.
2. `02_spread_modeling.ipynb` — compute log prices, hedge model, spread, and z-score.
3. `03_backtest_report.ipynb` — run backtest logic and summarize performance metrics.

This notebook has 8 sections as follows:

 1. Project Setup and Imports

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import os
import matplotlib.pyplot as plt

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import fetch_tiingo_price_matrix

print(PROJECT_ROOT)

2. Load Environment variables

In [ ]:
load_dotenv(PROJECT_ROOT / ".env")
api_key = os.getenv("TIINGO_API_KEY")

3. Define tickers and date range. AAPL and MSFT are used as an initial large-cap technology pair for pipeline demonstration. Pair choice is illustrative in v1; later versions can add systematic pair selection using correlation, cointegration, sector grouping, and stability checks.

In [ ]:
tickers=tickers = [
    "AAPL", "MSFT", "GOOGL", "META", "AMZN",
    "NVDA", "ADBE", "CRM", "ORCL", "INTC"
]
start_date = '2020-01-01'
end_date = '2024-12-31'
price_field = 'adjClose'
volume_field = 'adjVolume'

4. Fetch prices using fetch_tiingo_price_matrix()

In [ ]:
price_matrix, volume_matrix, loader_diagnostics = fetch_tiingo_price_matrix(tickers,start_date,end_date,api_key=api_key,price_field=price_field,volume_field=volume_field,validate=True)

5. Inspect price matrix and volume matrix

In [ ]:
price_matrix.head()

In [ ]:
price_matrix.shape

In [ ]:
volume_matrix.head()

In [ ]:
volume_matrix.shape

In [ ]:
print(price_matrix.index.equals(volume_matrix.index))
print(price_matrix.columns.equals(volume_matrix.columns))

In [ ]:
dollar_volume_matrix = price_matrix * volume_matrix
print(dollar_volume_matrix.head())
print(dollar_volume_matrix.shape)

6. Inspect loader diagnostics

In [ ]:
print(loader_diagnostics.keys())

In [ ]:
print("Data Loader Checks:-",loader_diagnostics['data_loader_checks_passed'])
print("Price Matrix Statistics:-",loader_diagnostics['price_matrix_summary'])
print("Price Matrix Validation:-",loader_diagnostics['price_validation_diagnostics'])
print("Volume Matrix Statistics:-",loader_diagnostics['volume_matrix_summary'])
print("Volume Matrix Validation:-",loader_diagnostics['volume_validation_diagnostics'])

In [ ]:
print("Ticker Diagnostics:-",pd.DataFrame(loader_diagnostics['ticker_loader_diagnostics']))

7. Save price_matrix to data/processed

In [ ]:
#Define filepath
price_file_path = PROJECT_ROOT/f"data/processed/price_matrix_tech_{pd.Timestamp(start_date).strftime('%Y')}_{pd.Timestamp(end_date).strftime('%Y')}.csv"
volume_file_path = PROJECT_ROOT/f"data/processed/volume_matrix_tech_{pd.Timestamp(start_date).strftime('%Y')}_{pd.Timestamp(end_date).strftime('%Y')}.csv"
dollar_matrix_file_path = PROJECT_ROOT/f"data/processed/dollar_volume_matrix_tech_{pd.Timestamp(start_date).strftime('%Y')}_{pd.Timestamp(end_date).strftime('%Y')}.csv"

#Ensure that the directory exists
os.makedirs(os.path.dirname(price_file_path), exist_ok=True)
os.makedirs(os.path.dirname(volume_file_path), exist_ok=True)
os.makedirs(os.path.dirname(dollar_matrix_file_path), exist_ok=True)

#Save the DataFrame
price_matrix.to_csv(price_file_path, index=True, encoding='utf-8')
volume_matrix.to_csv(price_file_path, index=True, encoding='utf-8')
dollar_volume_matrix.to_csv(price_file_path, index=True, encoding='utf-8')

8. Plot normalized prices.

In [ ]:
normalized_prices = price_matrix/price_matrix.iloc[0]
normalized_prices.plot(figsize=(12,6))
plt.title('Normalized price values from 2020 to 2024')
plt.xlabel('Date')
plt.ylabel('Normalized prices')
plt.legend(title="Ticker")
plt.grid(True)
plt.show()


The notebook is intentionally kept thin: core logic is implemented in `src/data_loader.py` and `src/data_validator.py`, while this notebook demonstrates usage and inspects outputs.